# LSCI220 Assignment 2: Scientific Texts vs. Literary Fiction
Noppachon Chaisongkhram | 300607581


## 1. Introduction
This work investigates differences in linguistic profiles between two text categories: "Science - Physics" (from Science & Technology category) and "Science-Fiction & Fantasy" (from Literature category). The research question is: "Do scientific texts exhibit greater lexical complexity and more neutral sentiment compared to literary fiction?"

The corpus consists of two distinct categories from the provided classification: (retrieved from the Project Gutenberg online library)
Scientific Texts (10 texts): (Physics texts representing the "Science & Technology" category | `LoC Class "QC: Science: Physics"`)
1. Carmilla
2. Dracula
3. Frankenstein
4. Gulliver's Travels into Several Remote Nations of the World
5. Rip Van Winkle
6. The call of Cthulhu
7. The King in Yellow
8. The Time Machine
9. The War of the Worlds
10. The Wonderful Wizard of Oz

Literary Fiction (10 texts): (Fantasy novels representing the "Literature" category)
1. Color Standards and Color Nomenclature
2. Experiments and Considerations Touching Colours (1664)
3. Goethe's Theory of Colours
4. Meteorology: The Science of the Atmosphere
5. Opticks
6. The A B C of Relativity
7. The Principle of Relativity
8. The Splash of a Drop
9. Treatise on light
10. Trinity Site

These categories are comparable because they represent fundamentally different communicative purposes: scientific communication (objective, factual) vs. literary expression (subjective, artistic).

Based on logical assumptions about these text categories:
1. Scientific texts should use more specialised, less frequent vocabulary with higher lexical sophistication
2. Scientific texts should have higher readability scores (more complex sentence structures)
3. Literary fiction should show more emotional variation and potentially more positive/negative sentiment
4. Literary fiction might have different lexical diversity patterns due to narrative style

Lexical profiles will be calculated via Word Frequency, Gunning Fog, Sentiment, and Type-Token Ratio (TTR).
* **Word Frequency** was chosen because it shows how common or rare words are, helping us see if texts use everyday language or specialised vocabulary.
* **Gunning Fog Index** was chosen because it measures how difficult a text is to read, based on sentence length and complex words.
* **Sentiment Analysis** was chosen because it reveals the emotional tone of a text, from negative to positive.
* **Type-Token Ratio (TTR)** was chosen because it measures vocabulary diversity by comparing unique words to total words.

## 2. Text Loading and Basic Setup

In [20]:
import nltk
import os
import statistics
import syllables  # Syllable counting (Workshop 9)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Download NLTK resources (Workshop 10)
nltk.download(['punkt_tab', 'vader_lexicon'])

# Sentiment analysis (Workshop 10)
sid = SentimentIntensityAnalyzer()

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jesse\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\jesse\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [21]:
# Load frequency dictionary (Workshop 8)
wf_dict = {}
with open('subtlxus_frequency.txt', 'r') as f:  # Read
    for line in f:
        parts = line.strip().split('\t')  # Clean by removing lead/trailing whitespaces and split by tab
        if len(parts) == 2:
            wf_dict[parts[0]] = float(parts[1])

print(f"Successfully loaded frequency dictionary with {len(wf_dict)} words.")  # wf_dict[word] = frequency_value

Successfully loaded frequency dictionary with 60384 words.


In [22]:
# List of text files
fiction_files = [
    "Carmilla.txt",
    "Dracula.txt",
    "Frankenstein.txt",
    "Gulliver's Travels into Several Remote Nations of the World.txt",
    "Rip Van Winkle.txt",
    "The call of Cthulhu.txt",
    "The King in Yellow.txt",
    "The Time Machine.txt",
    "The War of the Worlds.txt",
    "The Wonderful Wizard of Oz.txt"
]

physics_files = [
    "Color Standards and Color Nomenclature.txt",
    "Experiments and Considerations Touching Colours (1664).txt",
    "Goethe's Theory of Colours.txt",
    "Meteorology.txt",
    "Opticks.txt",
    "The A B C of Relativity.txt",
    "The Principle of Relativity.txt",
    "The Splash of a Drop.txt",
    "Treatise on light.txt",
    "Trinity Site.txt"
]

# Check files exist
print("Checking fiction files:")
for file in fiction_files:
    if os.path.exists("fiction/" + file):
        print(f"FOUND - {file}")
    else:
        print(f"NOT FOUND - {file}")

print("\nChecking physics files:")
for file in physics_files:
    if os.path.exists("physics/" + file):
        print(f"FOUND - {file}")
    else:
        print(f"NOT FOUND - {file}")

Checking fiction files:
FOUND - Carmilla.txt
FOUND - Dracula.txt
FOUND - Frankenstein.txt
FOUND - Gulliver's Travels into Several Remote Nations of the World.txt
FOUND - Rip Van Winkle.txt
FOUND - The call of Cthulhu.txt
FOUND - The King in Yellow.txt
FOUND - The Time Machine.txt
FOUND - The War of the Worlds.txt
FOUND - The Wonderful Wizard of Oz.txt

Checking physics files:
FOUND - Color Standards and Color Nomenclature.txt
FOUND - Experiments and Considerations Touching Colours (1664).txt
FOUND - Goethe's Theory of Colours.txt
FOUND - Meteorology.txt
FOUND - Opticks.txt
FOUND - The A B C of Relativity.txt
FOUND - The Principle of Relativity.txt
FOUND - The Splash of a Drop.txt
FOUND - Treatise on light.txt
FOUND - Trinity Site.txt


## 3. Linguistic Profiles

In [23]:
# Average word frequency (Workshop 8)
def get_word_freq(text):
    """Calculate average word frequency for a text"""
    # Tokenise and clean text
    tokens = nltk.word_tokenize(text.lower())
    words = [word for word in tokens if word.isalpha()]  # Only letters

    # Get frequencies for words in dictionary
    freqs = []
    for word in words:
        if word in wf_dict:
            freqs.append(wf_dict[word])

    if len(freqs) == 0:
        return 0

    return statistics.mean(freqs)

# Gunning Fog Index (Workshop 9)
def get_gunning_fog(text):
    """Calculate Gunning Fog readability score"""
    # Split into sentences
    sentences = nltk.sent_tokenize(text)

    # Get words
    words = [word for word in nltk.word_tokenize(text) if word.isalpha()]

    # Count complex words (3+ syllables)
    complex_words = 0
    for word in words:
        try:
            if syllables.estimate(word) >= 3:
                complex_words += 1
        except:
            # Simple vowel count as backup
            vowels = 'aeiou'
            vowel_count = sum(1 for char in word.lower() if char in vowels)
            if vowel_count >= 3:
                complex_words += 1

    # Avoid division by zero
    if len(sentences) == 0 or len(words) == 0:
        return 0

    # Calculate the index (GF formula)
    words_per_sentence = len(words) / len(sentences)
    percent_complex = (complex_words / len(words)) * 100
    fog_index = 0.4 * (words_per_sentence + percent_complex)

    return fog_index

# Average sentiment (Workshop 10)
def get_sentiment(text):
    """Calculate average sentiment score"""
    sentences = nltk.sent_tokenize(text)

    if len(sentences) == 0:
        return 0

    scores = []
    for sentence in sentences:
        if len(sentence.strip()) > 0:
            score = sid.polarity_scores(sentence)['compound']
            scores.append(score)

    if len(scores) == 0:
        return 0

    return statistics.mean(scores)

# Type-Token Ratio (Workshop 2)
def get_ttr(text):
    """Calculate Type-Token Ratio"""
    tokens = nltk.word_tokenize(text.lower())
    words = [word for word in tokens if word.isalpha()]

    if len(words) == 0:
        return 0

    types = set(words)
    return len(types) / len(words)

# Get all measures
def analyse_text(text, name=""):
    """Calculate all four measures for a text"""
    print(f"  Analysing {name}...")

    results = {
        'word_freq': get_word_freq(text),
        'gunning_fog': get_gunning_fog(text),
        'sentiment': get_sentiment(text),
        'ttr': get_ttr(text)
    }

    return results

## 4. Analysis and Results

### Fiction:

In [24]:
# Analyse fiction texts
print("ANALYSING FICTION TEXTS")
print("=" * 50)

fiction_results = []

for filename in fiction_files:
    try:
        # Read file
        with open("fiction/" + filename, 'r', encoding='utf-8') as f:
            text = f.read()

        # Get short name for display
        short_name = filename.replace('.txt', '')[:25]

        # Analyse text
        results = analyse_text(text, short_name)
        results['name'] = short_name
        fiction_results.append(results)

        # Print results for this text
        print(f"    Word Frequency: {results['word_freq']:.2f}")
        print(f"    Gunning Fog: {results['gunning_fog']:.1f}")
        print(f"    Sentiment: {results['sentiment']:.3f}")
        print(f"    TTR: {results['ttr']:.3f}")
        print()

    except Exception as e:
        print(f"Error analysing {filename}: {e}")

print(f"\nAnalysed {len(fiction_results)} fiction texts")

ANALYSING FICTION TEXTS
  Analysing Carmilla...
    Word Frequency: 4.51
    Gunning Fog: 14.6
    Sentiment: 0.089
    TTR: 0.140

  Analysing Dracula...
    Word Frequency: 4.69
    Gunning Fog: 11.8
    Sentiment: 0.049
    TTR: 0.057

  Analysing Frankenstein...
    Word Frequency: 4.40
    Gunning Fog: 16.0
    Sentiment: 0.043
    TTR: 0.092

  Analysing Gulliver's Travels into S...
    Word Frequency: 4.46
    Gunning Fog: 21.0
    Sentiment: 0.159
    TTR: 0.076

  Analysing Rip Van Winkle...
    Word Frequency: 4.35
    Gunning Fog: 17.1
    Sentiment: 0.076
    TTR: 0.222

  Analysing The call of Cthulhu...
    Word Frequency: 4.29
    Gunning Fog: 17.6
    Sentiment: -0.008
    TTR: 0.232

  Analysing The King in Yellow...
    Word Frequency: 4.49
    Gunning Fog: 12.2
    Sentiment: 0.029
    TTR: 0.109

  Analysing The Time Machine...
    Word Frequency: 4.45
    Gunning Fog: 12.4
    Sentiment: 0.025
    TTR: 0.137

  Analysing The War of the Worlds...
    Word Frequency:

### Physics:

In [25]:
# Analyse physics texts
print("\nANALYSING PHYSICS TEXTS")
print("=" * 50)

physics_results = []

for filename in physics_files:
    try:
        # Read file
        with open("physics/" + filename, 'r', encoding='utf-8') as f:
            text = f.read()

        # Get short name for display
        short_name = filename.replace('.txt', '')[:25]

        # Analyse text
        results = analyse_text(text, short_name)
        results['name'] = short_name
        physics_results.append(results)

        # Print results for this text
        print(f"    Word Frequency: {results['word_freq']:.2f}")
        print(f"    Gunning Fog: {results['gunning_fog']:.1f}")
        print(f"    Sentiment: {results['sentiment']:.3f}")
        print(f"    TTR: {results['ttr']:.3f}")
        print()

    except Exception as e:
        print(f"Error analysing {filename}: {e}")

print(f"\nAnalysed {len(physics_results)} physics texts")


ANALYSING PHYSICS TEXTS
  Analysing Color Standards and Color...
    Word Frequency: 3.91
    Gunning Fog: 19.0
    Sentiment: 0.032
    TTR: 0.146

  Analysing Experiments and Considera...
    Word Frequency: 4.47
    Gunning Fog: 26.9
    Sentiment: 0.207
    TTR: 0.069

  Analysing Goethe's Theory of Colour...
    Word Frequency: 4.30
    Gunning Fog: 16.1
    Sentiment: 0.126
    TTR: 0.076

  Analysing Meteorology...
    Word Frequency: 4.25
    Gunning Fog: 18.2
    Sentiment: 0.090
    TTR: 0.085

  Analysing Opticks...
    Word Frequency: 4.42
    Gunning Fog: 19.7
    Sentiment: 0.119
    TTR: 0.044

  Analysing The A B C of Relativity...
    Word Frequency: 4.52
    Gunning Fog: 16.4
    Sentiment: 0.084
    TTR: 0.077

  Analysing The Principle of Relativi...
    Word Frequency: 4.44
    Gunning Fog: 17.1
    Sentiment: 0.122
    TTR: 0.067

  Analysing The Splash of a Drop...
    Word Frequency: 4.33
    Gunning Fog: 12.6
    Sentiment: 0.041
    TTR: 0.189

  Analysing Tr

In [26]:
# Calculate category averages
def calculate_averages(results_list, category_name):
    """Calculate average scores for a category"""
    if len(results_list) == 0:
        return {}

    # Get all values for each measure
    word_freqs = [r['word_freq'] for r in results_list]
    fog_scores = [r['gunning_fog'] for r in results_list]
    sentiments = [r['sentiment'] for r in results_list]
    ttrs = [r['ttr'] for r in results_list]

    # Calculate averages
    averages = {
        'category': category_name,
        'avg_word_freq': statistics.mean(word_freqs),
        'avg_gunning_fog': statistics.mean(fog_scores),
        'avg_sentiment': statistics.mean(sentiments),
        'avg_ttr': statistics.mean(ttrs)
    }

    return averages

# Get averages for both categories
fiction_avg = calculate_averages(fiction_results, "Fiction")
physics_avg = calculate_averages(physics_results, "Physics")

# Display results
print("\n" + "=" * 70)
print("CATEGORY AVERAGES")
print("=" * 70)

print(f"\n{'Measure':<25} {'Fiction':<15} {'Physics':<15} {'Difference':<15}")
print("-" * 70)

for measure_key, measure_name in [
    ('avg_word_freq', 'Average Word Frequency'),
    ('avg_gunning_fog', 'Gunning Fog Index'),
    ('avg_sentiment', 'Average Sentiment'),
    ('avg_ttr', 'Type-Token Ratio')
]:
    fiction_val = fiction_avg[measure_key]
    physics_val = physics_avg[measure_key]
    diff = physics_val - fiction_val

    print(f"{measure_name:<25} {fiction_val:<15.3f} {physics_val:<15.3f} {diff:<+15.3f}")

print("-" * 70)



CATEGORY AVERAGES

Measure                   Fiction         Physics         Difference     
----------------------------------------------------------------------
Average Word Frequency    4.475           4.340           -0.134         
Gunning Fog Index         14.798          18.208          +3.410         
Average Sentiment         0.050           0.098           +0.048         
Type-Token Ratio          0.125           0.107           -0.018         
----------------------------------------------------------------------


**Word Frequency:** Lower = rarer words (more sophisticated vocabulary)

**Gunning Fog:** Higher = more complex text (harder to read)

**Sentiment:** Positive = more positive language, Negative = more negative

**TTR:** Higher = more diverse vocabulary

## 5. Critical Reflection

The analysis of linguistic profiles between fiction and physics texts gave some expected and some surprising results. Overall, the findings largely support the initial research questions, though with interesting points that should be discussed.

Looking at the results, the first hypothesis that physics texts would use more specialised vocabulary is supported, though the difference is modest. Physics texts show a slightly lower average word frequency (4.340 vs. 4.475), indicating they do indeed employ more technical or less common terminology. This aligns with what one might expect from scientific papers, which often require precise terminology not found in everyday language. The relatively small difference, however, suggests that both genres share a substantial portion of their vocabulary, perhaps because physics texts still rely on common function words, and that basic scientific terms have become more familiar over time.

The second hypothesis receives stronger support, with physics texts demonstrating notably higher Gunning Fog scores (18.208 vs. 14.798). This substantial difference confirms that physics texts are indeed more syntactically complex, featuring longer sentences and more intricate grammatical structures. This complexity reflects the need for precision in scientific writing, where qualifications, conditions, and relationships between concepts must be carefully articulated. This difference suggests readability is one of the clearest distinctions between these text types.

The sentiment analysis results present an interesting finding that somewhat contradicts the third hypothesis. Contrary to expectations, physics texts show slightly higher average sentiment scores (0.098 vs. 0.050). However, while both scores are relatively close to neutral, the finding challenges the assumption that scientific writing is devoid of emotional content. This might be explained by the fact that scientific texts express enthusiasm for discovery or positive anticipation of results. Alternatively, it could reflect limitations in the sentiment analysis tool, which might interpret certain scientific terminology in unexpected ways.

The fourth hypothesis regarding lexical diversity shows minimal difference, with fiction texts displaying only slightly higher Type-Token Ratios (0.125 vs. 0.107). This suggests that vocabulary range is not a strong distinguishing factor between these genres. The small difference indicates that while the specific vocabulary differs between fiction and physics, the overall variety of words used is comparable. This finding highlights that both types of writing employ diverse language to achieve their respective purposes, whether that be narrative development or conceptual explanation.

For future research, analysing contemporary scientific articles alongside modern fiction would provide a more current comparison. Employing additional measures such as Flesch Reading Ease or specific emotion word categories, would offer a more comprehensive linguistic profile. Investigating whether these patterns hold across different scientific disciplines or fiction genres would also be good.

In conclusion, this analysis reveals meaningful differences in how language is used across different text types. While physics texts are more syntactically complex as expected, the similarities in vocabulary diversity and the unexpected sentiment findings suggest that genre distinctions are not always straightforward. The study shows how quantitative analysis can both confirm assumptions and reveal unexpected patterns, providing a better understanding of how language varies according to communicative purpose.

This work is publicly accessible via GitHub: https://github.com/jescsk/LSCI220/tree/main/A2

*Code complexity toned down from first assignment, conclusions are more connected, based off feedback.*
